In [ ]:
!pip install -q yt-dlp pytchat opencv-python-headless numpy Pillow

In [ ]:
import os
import time
import tempfile
import yt_dlp
import pytchat
import cv2
import numpy as np
from typing import List, Tuple

class StreamFetcher:
    def __init__(self, url: str, fps: float = 1.0):
        self.url = url
        self.fps = fps
        try:
            self.video_id = url.split("v=")[1].split("&")[0]
        except (IndexError, AttributeError):
            self.video_id = url.split("/")[-1]
        self.chat = None
        self.temp_dir = tempfile.mkdtemp()

    def start(self):
        """Initializes the connection to the chat stream."""
        self.chat = pytchat.create(video_id=self.video_id)

    def get_data_window(self, duration_sec: int) -> Tuple[List[str], List[np.ndarray]]:
        """
        Fetches 'duration_sec' worth of chat messages and captures frames.
        Returns: (chat_messages, list_of_numpy_frames)
        """
        start_time = time.time()
        chat_messages = []
        frames = []

        # 1. Get the direct stream URL
        ydl_opts = {"format": "best[height<=360]", "quiet": True}
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            try:
                info = ydl.extract_info(self.url, download=False)
                stream_url = info["url"]
            except Exception as e:
                print(f"Error extracting stream URL: {e}")
                return [], []

        # 2. Open stream with OpenCV
        cap = cv2.VideoCapture(stream_url)

        # 3. Poll chat and capture frames
        while time.time() - start_time < duration_sec:
            # Capture frame
            ret, frame = cap.read()
            if ret:
                # Capture at requested FPS
                if int((time.time() - start_time) * self.fps) > len(frames):
                    # Convert BGR to RGB
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(rgb_frame)

            # Poll chat
            if self.chat and self.chat.is_alive():
                for c in self.chat.get().sync_items():
                    chat_messages.append(f"{c.author.name}: {c.message}")

            time.sleep(0.01)

        cap.release()

        # Final chat drain
        if self.chat and self.chat.is_alive():
            for c in self.chat.get().sync_items():
                chat_messages.append(f"{c.author.name}: {c.message}")

        return chat_messages, frames

    def stop(self):
        if self.chat:
            self.chat.terminate()

In [ ]:
VIDEO_SOURCES = {
    "Music & Study": [
        {"name": "Lofi Girl", "url": "https://www.youtube.com/watch?v=jfKfPfyJRdk"},
        {"name": "Monstercat Silk", "url": "http://www.youtube.com/watch?v=WsDyRAPFBC8"},
        {"name": "Chillhop Music", "url": "http://www.youtube.com/watch?v=7NOSDKb0HlU"},
    ],
    "Nature & Animals": [
        {"name": "Explore.org Puppy Cam", "url": "http://www.youtube.com/watch?v=h-Z0wCdD3dI"},
        {"name": "iPanda", "url": "http://www.youtube.com/watch?v=9LvjI3NelAU"},
        {"name": "NamibiaCam", "url": "https://www.youtube.com/watch?v=ydYDqZQpim8"},
        {"name": "AfriCam", "url": "https://www.youtube.com/watch?v=qpukdDslCjk"},
        {"name": "Wildlife In The Forest", "url": "https://www.youtube.com/watch?v=F0GOOP82094"},
    ],
    "News & Finance": [
        {"name": "Sky News", "url": "http://www.youtube.com/watch?v=YDvsBbKfLPA"},
        {"name": "Al Jazeera English", "url": "http://www.youtube.com/watch?v=YDvsBbKfLPA"},
        {"name": "Bloomberg Television", "url": "http://www.youtube.com/watch?v=iEpJwprxDdk"},
    ],
    "Space & Science": [
        {"name": "NASA Live", "url": "http://www.youtube.com/watch?v=m3kR2KK8TEs"},
        {"name": "Blooket Live", "url": "http://www.youtube.com/watch?v=M5OKNwczOP8"},
    ],
    "Urban & Transprot": [
        {"name": "La Plata", "url": "https://www.youtube.com/watch?v=X-ir2KfXMX0"},
        {"name": "Tokyo Streets", "url": "https://www.youtube.com/watch?v=L6wO1-U2RTY"},
    ],
}

In [ ]:
import time
import os
import textwrap
import numpy as np
import kaggle_benchmarks as kbench
from typing import List, Tuple
from PIL import Image

@kbench.task(name="past_frame_generation", version=2)
def past_frame_generation(llm) -> float:
    """
    Evaluates the AGI's ability to reconstruct the visual state of a 10s segment
    given the previous 30s context and ONLY the chat logs from the withheld 10s interval.
    """
    all_videos = [vid['url'] for category in VIDEO_SOURCES.values() for vid in category]
    total_score = 0.0
    valid_evals = 0

    for video_url in all_videos:
        print(f"\n--- Evaluating Video: {video_url} ---")
        fetcher = StreamFetcher(video_url, fps=0.2)
        try:
            fetcher.start()
            
            print("Providing 30s of initial context...")
            context_chat_list, context_frames = fetcher.get_data_window(duration_sec=30)
            context_chat_text = "\n".join(context_chat_list)

            print("Capturing withheld segment (10s)...")
            withheld_chat_list, ground_truth_frames = fetcher.get_data_window(duration_sec=10)
            withheld_chat_text = "\n".join(withheld_chat_list)

            prompt = textwrap.dedent(f"""
                You are an AI expert in visual reconstruction.
                You have been watching a YouTube live stream. I have provided the last 30 seconds of video.
                
                Now, for the NEXT 10 seconds, the VIDEO IS WITHHELD. You only have the chat logs.

                --- PREVIOUS CHAT CONTEXT (0-30s) ---
                {context_chat_text}

                --- WITHHELD CHAT LOGS (30-40s) ---
                {withheld_chat_text}
                --- END CHAT ---

                Task: Reconstruct what happened visually during that withheld 10-second interval. 
                Describe exactly 10 frames (one for each second). 
                Include details on the environment, screen changes, and the actions occurring.
            """)
            
            pil_frames = [Image.fromarray(f) for f in context_frames]
            
            predicted_visuals = None
            for attempt in range(4):
                try:
                    predicted_visuals = llm.prompt([prompt, *pil_frames])
                    break
                except Exception as e:
                    if attempt < 3 and ('503' in str(e) or '429' in str(e) or 'unavailable' in str(e).lower()):
                        print(f"LLM API unavailable ({e}), retrying in 10s...")
                        time.sleep(10)
                    else:
                        raise

            criteria = [
                "The reconstructed descriptions align with the events described in the withheld chat.",
                "The temporal sequence of events matches the chat's reaction timing.",
                "The visual descriptions are plausible for the context of this specific stream."
            ]

            def judge_prompt_fn(criteria: list[str], response_text: str) -> str:
                formatted_criteria = "\n".join(f"- {c}" for c in criteria)
                return textwrap.dedent(f"""
                    The 'Predicted Reconstruction' was generated based on context and withheld chat.
                    Evaluate it against the 'Withheld Chat Context'.
                    
                    WITHHELD CHAT CONTEXT:
                    ```
                    {withheld_chat_text}
                    ```

                    PREDICTED RECONSTRUCTION:
                    ```
                    {response_text}
                    ```

                    CRITERIA:
                    {formatted_criteria}
                    
                    Did the AI successfully 'see' the video through the eyes of the chat?
                """)

            assessment = None
            for attempt in range(4):
                try:
                    assessment = kbench.assertions.assess_response_with_judge(
                        criteria=criteria,
                        response_text=predicted_visuals,
                        judge_llm=kbench.judge_llm,
                        prompt_fn=judge_prompt_fn
                    )
                    break
                except Exception as e:
                    if attempt < 3 and ('503' in str(e) or '429' in str(e) or 'unavailable' in str(e).lower()):
                        print(f"Judge API unavailable ({e}), retrying in 10s...")
                        time.sleep(10)
                    else:
                        raise

            if assessment is not None:
                successes = 0
                for result in assessment.results:
                    kbench.assertions.assert_true(
                        result.passed,
                        expectation=f"[{video_url}] Criterion '{result.criterion}' failed. Reason: {result.reason}"
                    )
                    if result.passed:
                        successes += 1

                score = (successes / len(criteria)) * 100.0
                total_score += score
                valid_evals += 1
                print(f"Score for this stream: {score:.2f}")
            else:
                print("Judge assessment failed for this stream.")

        except Exception as e:
            print(f"Failed processing {video_url}: {e}")
        finally:
            fetcher.stop()

    if valid_evals == 0:
        kbench.assertions.assert_fail("All 10 video evaluations failed.")
        return 0.0

    final_score = total_score / valid_evals
    print(f"\nFINAL SCORE ACROSS {valid_evals} STREAMS: {final_score:.2f}")
    return float(final_score)

if __name__ == "__main__":
    past_frame_generation.run(kbench.llm)